In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "projects" / "poreallas"
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import numpy as np
import pandas as pd
import xarray as xr
from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils, isku_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'utils.isku_utils' from '/home/emily_zuetell/projects/poreallas/utils/isku_utils.py'>

In [7]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

### Set this to the current projection Month (e.g. 10: October forecast)
PROJ_MONTH = 10
MONTHS = [
    (PROJ_MONTH - 1 + i) % 12 + 1 for i in range(6)
]

MONTH_LABELS = {10: "October", 11: "November", 12: "December",
                1: "January", 2: "February", 3: "March"}
MONTH_ORDER = list(MONTH_LABELS.values()) + ["6-month total"]

# Baseline period (Verify: 1996-2025)
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")

RATE = False  # [False: Total Deaths, True: Mortality Rate] -> Use False for Tracker

# Define Forecast
EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]

# Open Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [8]:
config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",  # Set version to today
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=RATE,
    months=MONTHS,  # Forecast Months (6 months)
    hotonly="hotonly",  # We are using only hot-only for the tracker
    dims=["number", "sample"],
)

In [9]:
impact = analysis_utils.compute_impact(
        effect.chunk({dim: -1 for dim in config.dims}), config, ensemble=True
    )
impact = impact.sel(month=config.months)
    


In [ ]:
filename_template="{version}_{hotonly}_{rate_l}_{group_level}.csv"

for group_level in ['IR', 'ISO', 'ADM1', 'global']:
# If you run into memory issues with ADM1, reduce "chunk_size" parameter in analysis_utils.py/redistribute_adm1
    if group_level == 'global':
        agg = analysis_utils.compute_global_impact(
                    impact, config.socioeconomics, rate=config.rate, group_dim="region"
                )
        base_cols = ["region"]
    else:
        agg, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, group_level)

    da = agg.sel(month=list(MONTH_LABELS)).chunk({'number': -1, 'sample': -1})
    
    dim = config.dims
    b = dim[1] if len(dim) > 1 else None

    # Per-month stats
    monthly = xr.Dataset({
        "mean": da.mean(dim=dim),
        "sem": analysis_utils.crossed_sem(da, a=dim[0], b=b),
    })

    # 6-month total: sum across months first, then compute stats
    total_da = da.sum("month")
    total = xr.Dataset({
        "mean": total_da.mean(dim=dim),
        "sem": analysis_utils.crossed_sem(total_da, a=dim[0], b=b),
    })

    df_month = analysis_utils.dataset_to_dataframe(monthly)
    df_month["month"] = df_month["month"].map(MONTH_LABELS)

    df_total = analysis_utils.dataset_to_dataframe(total)
    df_total["month"] = "6-month total"

    if group_level == 'global':
        df_month["region"] = "global"
        df_total["region"] = "global"

    long = pd.concat([df_month, df_total], ignore_index=True)
    long["month"] = pd.Categorical(long["month"], categories=MONTH_ORDER, ordered=True)
    long = (long.sort_values(base_cols + ["month"])
                [base_cols + ["month", "mean", "sem"]]
                .reset_index(drop=True))

    long_rounded = analysis_utils.round_output(long)

    fname = filename_template.format(
        version=config.version, 
        hotonly=config.hotonly, 
        rate_l="total",
        group_level=group_level,
    )
    long.to_csv(fname, index=False)
    long_rounded.to_csv(fname.replace(".csv", "_rounded.csv"), index=False)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
